# 02a — CLMS-Layer manuell herunterladen (Sprint 2a)

> **Lernziel Sprint 2a:**  
> Wer versteht, wie ein HTTP-Request gegen einen COG-Endpoint und das ArcGIS `exportImage`-API  
> funktioniert, versteht danach sofort *warum* `rioxarray.open_rasterio(url)` so effizient ist.  
> Hier wird bewusst kein High-Level-Client (rioxarray, OWSLib) genutzt — nur `requests` + `rasterio`.

## Was wird in diesem Notebook gemacht?

1. **Konfiguration:** Testgebiet, Ausgabepfade, alle 5 Layer-Definitionen
2. **CDSE STAC:** Metadaten-Suche für VLCC-Layer (TCD, Grassland)
3. **EEA REST `exportImage`:** Download für IMD (Imperviousness), WAW (Water & Wetness), FTY (Forest Type)
4. **CDSE OData-Download:** Download der VLCC-Kacheln via Produktlink (mit Auth)
5. **Inspektion:** Metadaten der heruntergeladenen Raster mit `rasterio` ausgeben
6. **Visualisierung:** Alle 5 Layer nebeneinander mit `matplotlib`
7. **HTTP-Mechanik unter der Haube:** Erklärung von HTTP Range Requests & COG-Struktur

---
**Testgebiet:** Nationalpark Donau-Auen (Natura-2000 AT1205A00)  
**Bounding Box (WGS84):** `[16.5, 48.08, 16.95, 48.22]` — `[minLon, minLat, maxLon, maxLat]`  
**Quellen:**
- CDSE STAC: `https://catalogue.dataspace.copernicus.eu/stac/` (VLCC-Layer, neueste Generation)
- EEA DiscoMap: `https://image.discomap.eea.europa.eu/arcgis/rest/services/GioLandPublic/` (HRL 2018)

## 0 — Imports & Setup

In [ ]:
# --- Standardbibliotheken ---
import os
import time
import zipfile
import shutil
from pathlib import Path

# --- Drittanbieter ---
import requests
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Lade Credentials aus .env (keine Secrets im Notebook!)
load_dotenv()

print('Imports OK')

## 1 — Konfiguration: Testgebiet, Pfade, Layer-Definitionen

Alle Layer sind hier als **Single Source of Truth** definiert. Sprint 2b wird diese
Konfiguration nach `src/biodiv_horizon/config.py` refaktorieren.

### Die 5 CLMS-Layer und ihre ökologische Bedeutung

| Layer | Kürzel | Einheit | Quelle | Ökologische Relevanz |
|:---|:---|:---|:---|:---|
| Imperviousness Density 2018 | IMD | % (0–100) | EEA REST | Versiegelungsgrad = Habitat-Destruktor |
| Water & Wetness 2018 | WAW | Klasse 0/1/2 | EEA REST | Auen-FFH-LRT, Schlüsselhabitat |
| Forest Type 2018 | FTY | Klasse 0/1/2 | EEA REST | Hartholzaue vs. Nadelforst |
| Tree Cover Density 2024 | TCD | % (0–100) | CDSE STAC | Kronendichte Auwald, Habitat Eisvogel |
| Grassland 2024 | GRA | Binär 0/1 | CDSE STAC | Au-Wiesen, Habitat Neuntöter & Feldhase |

In [ ]:
# ---------------------------------------------------------------------------
# Testgebiet: Nationalpark Donau-Auen (BBox in WGS84 / EPSG:4326)
# Konvention GIS: [minLon (West), minLat (Süd), maxLon (Ost), maxLat (Nord)]
# Bei WCS/REST-APIs: bbox-Parameter = 'minx,miny,maxx,maxy'
# ---------------------------------------------------------------------------
BBOX = [16.5, 48.08, 16.95, 48.22]   # [minLon, minLat, maxLon, maxLat]
BBOX_STR = ','.join(str(v) for v in BBOX)  # '16.5,48.08,16.95,48.22'
BBOX_SR   = 4326                            # Spatial Reference EPSG-Code

# Ausgabeverzeichnis
OUTPUT_DIR = Path('../data/raw/copernicus')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CDSE Credentials (aus .env geladen)
CDSE_USER = os.getenv('CDSE_USERNAME', '')
CDSE_PASS = os.getenv('CDSE_PASSWORD', '')
HAS_CDSE_CREDENTIALS = bool(CDSE_USER and CDSE_PASS)

print(f'Testgebiet: Nationalpark Donau-Auen')
print(f'  BBox (WGS84): {BBOX_STR}')
print(f'  Ausgabepfad:  {OUTPUT_DIR.resolve()}')
print(f'  CDSE-Account: {"gefunden" if HAS_CDSE_CREDENTIALS else "NICHT in .env — nur EEA-Download"}')

In [ ]:
# ---------------------------------------------------------------------------
# Layer-Definitionen — Single Source of Truth für Sprint 2a
#
# source == 'eea_rest'   → EEA DiscoMap ArcGIS ImageServer (exportImage-API)
# source == 'cdse_stac'  → CDSE STAC + OData-Download (OAuth2-Auth)
# ---------------------------------------------------------------------------
CLMS_LAYERS = {
    'IMD': {
        'description': 'Imperviousness Density 2018 (Versiegelungsgrad)',
        'unit': '% (0-100)',
        'colormap': 'YlOrRd',
        'source': 'eea_rest',
        'eea_service': 'HRL_ImperviousnessDensity_2018',
        'filename': 'imd_2018_donau_auen.tif',
    },
    'WAW': {
        'description': 'Water and Wetness 2018 (Gewaesser & Feuchtgebiete)',
        'unit': 'Klasse (0=trocken, 1=feucht, 2=Wasser)',
        'colormap': 'Blues',
        'source': 'eea_rest',
        'eea_service': 'HRL_WaterWetness_2018',
        'filename': 'waw_2018_donau_auen.tif',
    },
    'FTY': {
        'description': 'Forest Type 2018 (Waldtyp)',
        'unit': 'Klasse (0=kein Wald, 1=Laub, 2=Nadel)',
        'colormap': 'Greens',
        'source': 'eea_rest',
        'eea_service': 'HRL_ForestType_2018',
        'filename': 'fty_2018_donau_auen.tif',
    },
    'TCD': {
        'description': 'Tree Cover Density 2024 (Kronendichte)',
        'unit': '% (0-100)',
        'colormap': 'YlGn',
        'source': 'cdse_stac',
        'stac_collection': 'clms_vlcc_tree-cover-density_europe_10m_yearly_v1',
        'filename': 'tcd_2024_donau_auen.tif',
    },
    'GRA': {
        'description': 'Grassland 2024 (Gruenland)',
        'unit': 'Binaer (0=kein Gruenland, 1=Gruenland)',
        'colormap': 'RdYlGn',
        'source': 'cdse_stac',
        'stac_collection': 'clms_vlcc_grassland_europe_10m_yearly_v1',
        'filename': 'gra_2024_donau_auen.tif',
    },
}

print(f'{len(CLMS_LAYERS)} Layer konfiguriert:')
for key, cfg in CLMS_LAYERS.items():
    src = 'EEA REST' if cfg['source'] == 'eea_rest' else 'CDSE STAC'
    print(f'  [{src}] {key}: {cfg["description"]}')

## 2 — CDSE STAC: Metadaten-Suche für VLCC-Layer

### Was ist STAC?

STAC (SpatioTemporal Asset Catalog) ist ein offener REST/JSON-Standard für Geodaten-Kataloge.
Ein STAC-**Item** ist ein GeoJSON-Feature mit:
- `id` — Eindeutiger Bezeichner (z.B. `CLMS_HRLVLCC_TCD_S2024_R10m_E48N28_03035_V01_R00`)
- `bbox` — Bounding Box des Items
- `properties.datetime` — Zeitstempel
- `assets` — Dictionary mit Download-/Streaming-URLs

### STAC-Abfrage

```
GET /stac/collections/{collection}/items?bbox=minLon,minLat,maxLon,maxLat&limit=N
→ GeoJSON FeatureCollection mit STAC Items
```

In [ ]:
CDSE_STAC_BASE = 'https://catalogue.dataspace.copernicus.eu/stac'

def search_stac_items(collection: str, bbox: list, limit: int = 5) -> list:
    """
    Durchsucht den CDSE-STAC-Katalog nach Items die die BBox ueberlappen.

    Args:
        collection: STAC Collection ID
        bbox: [minLon, minLat, maxLon, maxLat] in WGS84
        limit: Maximale Anzahl Items

    Returns:
        Liste von STAC Item-Dictionaries
    """
    url = f'{CDSE_STAC_BASE}/collections/{collection}/items'
    params = {'bbox': ','.join(str(v) for v in bbox), 'limit': limit}

    print(f'  GET {url.split("/stac/")[1]}')
    print(f'      bbox={params["bbox"]}')

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    items = r.json().get('features', [])
    print(f'      HTTP {r.status_code} -> {len(items)} Items')
    return items


# Suche fuer die CDSE-STAC-Layer (TCD, GRA)
stac_results = {}
for key, cfg in CLMS_LAYERS.items():
    if cfg['source'] != 'cdse_stac':
        continue
    print(f'\nSuche {key}: {cfg["description"]}...')
    stac_results[key] = search_stac_items(cfg['stac_collection'], BBOX)

In [ ]:
# STAC-Item-Anatomie: Was steckt in einem Item?
print('=' * 65)
print('Anatomy eines STAC-Items (Beispiel: TCD)')
print('=' * 65)

if stac_results.get('TCD'):
    item = stac_results['TCD'][0]
    print(f'\nID:       {item["id"]}')
    print(f'Datum:    {item["properties"].get("datetime", "n/a")}')
    print(f'BBox:     {item["bbox"]}')

    # Kachel-Nomenklatur erklaeren
    item_id = item['id']
    parts = item_id.split('_')
    print(f'\nKachel-ID Anatomie: {item_id}')
    print(f'  HRLVLCC = High Resolution Layer / Vegetation Land Cover Component')
    print(f'  TCD     = Tree Cover Density')
    print(f'  S2024   = Referenzjahr 2024')
    print(f'  R10m    = 10m raeumliche Aufloesung')
    print(f'  E48N28  = ETRS89-LAEA Kachel-Kennung (EPSG:3035)')
    print(f'  03035   = CRS EPSG:3035')

    print(f'\nAssets:')
    for akey, aval in item['assets'].items():
        href = aval.get('href', '')
        rtype = aval.get('type', 'unbekannt')
        print(f'  [{akey}] type={rtype}')
        print(f'    href: {href[:100]}...' if len(href) > 100 else f'    href: {href}')

    print('\nHinweis:')
    print('  "data"-Asset:    s3:// -> COG-Streaming via GDAL VSICURL (Sprint 2b)')
    print('  "product"-Asset: https:// -> OData-Download als ZIP (Sprint 2a, braucht Auth)')

## 3 — EEA DiscoMap REST: Download via `exportImage`

### Warum EEA REST und nicht STAC?

Die IMD (Imperviousness), WAW (Water & Wetness) und FTY (Forest Type) 2018-Layer
liegen auf der **EEA-ArcGIS-Infrastruktur (DiscoMap)**. Die OGC WCS-Extension ist
dort serverseitig deaktiviert (HTTP 400). Stattdessen nutzen wir den nativen
**ArcGIS `exportImage`-Endpunkt**, der direkt binäre GeoTIFFs für beliebige BBoxen liefert:

```
GET /arcgis/rest/services/GioLandPublic/{service}/ImageServer/exportImage
    ?bbox=minx,miny,maxx,maxy   <- WGS84-Koordinaten
    &bboxSR=4326                <- Spatial Reference der BBox
    &size=breite,hoehe          <- Ausgabe-Pixelgröße
    &format=tiff                <- Ausgabeformat
    &f=image                    <- Binärstream statt JSON
```

### Auflösungs-Berechnung

Die Donau-Auen-BBox ist ca. **33.75 km × 15.54 km** groß:
- `0.45° Lon × ~75 km/° ≈ 33.75 km → 33750m / 40m ≈ 844 Pixel (Breite)`
- `0.14° Lat × ~111 km/° ≈ 15.54 km → 15540m / 40m ≈ 388 Pixel (Höhe)`

Sprint 2a: **~40m** Auflösung (schneller Download, Lernzweck).  
Sprint 2b: Volle **10m** Auflösung (`size=3375,1554`).

In [ ]:
EEA_BASE = 'https://image.discomap.eea.europa.eu/arcgis/rest/services/GioLandPublic'

# Sprint 2a: ~40m Aufloesung fuer schnellen Download
IMG_W = 844
IMG_H = 388
print(f'Download-Aufloesung: {IMG_W} x {IMG_H} px (~40m GSD)')
print(f'Fuer volle 10m: IMG_W=3375, IMG_H=1554 (Sprint 2b)')
print()


def download_eea_layer(service_name: str, output_path: Path, bbox: list,
                       img_w: int = 844, img_h: int = 388, bbox_sr: int = 4326) -> Path:
    """
    Laedt einen EEA DiscoMap ImageServer-Layer als GeoTIFF herunter.

    Nutzt den ArcGIS-nativen exportImage-Endpunkt.
    Der Server schneidet serverseitig zu und liefert binaeren GeoTIFF-Strom.
    Kein manuelles Clipping notwendig!

    Args:
        service_name: ArcGIS-Servicename (z.B. 'HRL_ImperviousnessDensity_2018')
        output_path:  Zielpfad fuer das GeoTIFF
        bbox:         [minLon, minLat, maxLon, maxLat] (WGS84)
        img_w, img_h: Ausgabe-Pixelgroesse
        bbox_sr:      Spatial Reference als EPSG-Code

    Returns:
        Pfad zur gespeicherten GeoTIFF-Datei
    """
    url = f'{EEA_BASE}/{service_name}/ImageServer/exportImage'
    params = {
        'bbox': ','.join(str(v) for v in bbox),
        'bboxSR': bbox_sr,
        'size': f'{img_w},{img_h}',
        'format': 'tiff',
        'f': 'image',        # Binaerstrom statt JSON-Antwort
        'pixelType': 'U8',   # Unsigned 8-bit Integer (0-255)
        'noDataInterpretation': 'esriNoDataMatchAny',
    }

    short_url = f'{service_name}/ImageServer/exportImage'
    print(f'  GET .../{short_url}')
    print(f'      size={img_w}x{img_h}px | bbox={params["bbox"]}')

    t0 = time.time()
    r = requests.get(url, params=params, timeout=60)
    elapsed = time.time() - t0

    ct = r.headers.get('Content-Type', '')
    if r.status_code != 200 or ('tiff' not in ct and 'image' not in ct):
        print(f'  FEHLER: HTTP {r.status_code}, Content-Type={ct}')
        print(f'  Antwort: {r.content[:200]}')
        r.raise_for_status()

    size_kb = len(r.content) / 1024
    print(f'      HTTP {r.status_code} -> {size_kb:.1f} KB in {elapsed:.1f}s')

    output_path.write_bytes(r.content)
    print(f'      Gespeichert: {output_path.name}')
    return output_path

In [ ]:
# Download aller EEA REST-Layer
downloaded_files = {}

for key, cfg in CLMS_LAYERS.items():
    if cfg['source'] != 'eea_rest':
        continue

    out_path = OUTPUT_DIR / cfg['filename']

    if out_path.exists():
        size_kb = out_path.stat().st_size / 1024
        print(f'SKIP {key}: bereits vorhanden ({size_kb:.0f} KB)')
        downloaded_files[key] = out_path
        continue

    print(f'\nLade {key}: {cfg["description"]}...')
    p = download_eea_layer(cfg['eea_service'], out_path, BBOX, IMG_W, IMG_H)
    downloaded_files[key] = p

print(f'\nEEA-Layer fertig: {len([k for k,c in CLMS_LAYERS.items() if c["source"]=="eea_rest"])} Dateien')

## 4 — CDSE OData-Download: VLCC-Layer (TCD & Grassland)

### Das Problem mit S3-URLs

Die `data`-Assets der STAC-Items haben `s3://eodata/...`-URLs — das sind
interne Pfade auf dem CDSE-Objektspeicher. Um direkt darauf
zuzugreifen, braucht man S3-Credentials oder GDAL VSICURL mit Bearer-Token.

### Sprint 2a-Lösung: OData-Produkt-Download (HTTPS)

Jedes STAC-Item hat auch ein `product`-Asset mit einer **HTTPS OData-URL**:
```
https://download.dataspace.copernicus.eu/odata/v1/Products({uuid})/$value
```
Diese URL erfordert einen **CDSE Bearer-Token** (OAuth 2.0 Password Flow).
Das heruntergeladene Produkt ist eine komplette **Europa-Kachel als ZIP** (~200–500 MB).

### Sprint 2b Preview

```python
# Sprint 2b - so kurz wird es sein:
import rioxarray as rxr
# GDAL-VSICURL: automatische HTTP Range Requests - NUR benoetigte Kacheln!
ds = rxr.open_rasterio(s3_url_with_gdal_prefix, chunks={'x': 2048, 'y': 2048})
clipped = ds.rio.clip_box(*BBOX, crs='EPSG:4326')
```

In [ ]:
def get_cdse_token(username: str, password: str) -> str:
    """
    Holt einen OAuth 2.0 Bearer-Token vom CDSE Identity Provider.

    Nutzt den Resource Owner Password Credentials Grant (ROPC) Flow.
    Token-Lebensdauer: ca. 10 Minuten.

    Returns:
        access_token (JWT-String)
    """
    url = ('https://identity.dataspace.copernicus.eu'
           '/auth/realms/CDSE/protocol/openid-connect/token')
    payload = {
        'client_id': 'cdse-public',
        'username': username,
        'password': password,
        'grant_type': 'password',
    }
    print('  Token-Request an CDSE Identity Provider...')
    r = requests.post(url, data=payload, timeout=15)
    if r.status_code != 200:
        err = r.json().get('error_description', r.text[:200])
        print(f'  Auth fehlgeschlagen (HTTP {r.status_code}): {err}')
        r.raise_for_status()
    token = r.json()['access_token']
    print(f'  Bearer-Token OK (Laenge: {len(token)} Zeichen)')
    return token


def download_cdse_product(product_url: str, output_path: Path,
                          token: str, chunk_size: int = 1024 * 1024) -> Path:
    """
    Laedt ein CDSE OData-Produkt als ZIP-Archiv herunter (streaming).

    CDSE liefert CLMS-Produkte als ZIP-Archiv mit dem GeoTIFF und Metadaten.

    Args:
        product_url: OData-URL (muss auf /$value enden)
        output_path: Zielpfad fuer das Archiv
        token:       Gueltiger CDSE Bearer-Token
        chunk_size:  Streaming-Chunkgroesse in Bytes (Standard: 1 MB)
    """
    headers = {'Authorization': f'Bearer {token}'}
    print(f'  Download: {product_url[:80]}...')
    t0 = time.time()

    with requests.get(product_url, headers=headers, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get('Content-Length', 0))
        print(f'  Dateigroesse: {total / 1024 / 1024:.1f} MB')

        downloaded = 0
        with open(output_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)
                downloaded += len(chunk)
                if total > 0:
                    pct = downloaded / total * 100
                    if int(pct) % 20 == 0 and pct > 0:
                        print(f'  ... {pct:.0f}% ({downloaded/1024/1024:.0f}/{total/1024/1024:.0f} MB)', end='\r')

    elapsed = time.time() - t0
    size_mb = output_path.stat().st_size / 1024 / 1024
    print(f'\n  Fertig: {size_mb:.1f} MB in {elapsed:.0f}s')
    return output_path


def clip_raster_to_bbox(input_path: Path, output_path: Path,
                        bbox: list, bbox_crs: str = 'EPSG:4326') -> Path:
    """
    Schneidet ein GeoTIFF auf eine BBox zu (rasterio window-based).

    WICHTIG: Die BBox wird vom bbox_crs in das Raster-CRS transformiert.
    Das Raster wird NICHT umprojiziert - nur ausgeschnitten.
    """
    from rasterio.windows import from_bounds as window_from_bounds
    from pyproj import Transformer

    with rasterio.open(input_path) as src:
        raster_crs_str = src.crs.to_string()
        print(f'  Raster CRS: {raster_crs_str}')

        # BBox in Raster-CRS transformieren
        tr = Transformer.from_crs(bbox_crs, raster_crs_str, always_xy=True)
        minx, miny = tr.transform(bbox[0], bbox[1])
        maxx, maxy = tr.transform(bbox[2], bbox[3])

        window = window_from_bounds(minx, miny, maxx, maxy, transform=src.transform)
        data = src.read(window=window)
        profile = src.profile.copy()
        profile.update({
            'height': data.shape[1], 'width': data.shape[2],
            'transform': src.window_transform(window),
            'compress': 'DEFLATE', 'tiled': True,
            'blockxsize': 512, 'blockysize': 512,
        })
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(data)

    print(f'  Geclippt: {output_path.name} ({output_path.stat().st_size/1024:.0f} KB)')
    return output_path

In [ ]:
# Download der CDSE STAC-Layer (TCD, GRA)
# Benoetigt CDSE_USERNAME + CDSE_PASSWORD in der .env-Datei
# Registrierung kostenlos: https://dataspace.copernicus.eu/

if not HAS_CDSE_CREDENTIALS:
    print('Keine CDSE-Credentials in .env.')
    print('CDSE-Layer (TCD, Grassland) werden uebersprungen.')
    print('-> In .env eintragen: CDSE_USERNAME=... / CDSE_PASSWORD=...')
    print('-> Registrierung: https://dataspace.copernicus.eu/')
else:
    print('CDSE-Credentials vorhanden. Starte Authentifizierung...')
    try:
        token = get_cdse_token(CDSE_USER, CDSE_PASS)

        for key, cfg in CLMS_LAYERS.items():
            if cfg['source'] != 'cdse_stac':
                continue

            out_path = OUTPUT_DIR / cfg['filename']
            if out_path.exists():
                print(f'SKIP {key}: bereits vorhanden')
                downloaded_files[key] = out_path
                continue

            if not stac_results.get(key):
                print(f'Keine STAC-Items fuer {key}')
                continue

            item = stac_results[key][0]
            product_url = item['assets']['product']['href']
            if not product_url.endswith('/$value'):
                product_url += '/$value'

            print(f'\nLade {key}: {cfg["description"]}')
            print(f'  STAC Item: {item["id"]}')

            # Download als ZIP
            zip_path = OUTPUT_DIR / f'{key.lower()}_raw.zip'
            download_cdse_product(product_url, zip_path, token)

            # ZIP extrahieren
            print('  Extrahiere ZIP...')
            extract_dir = OUTPUT_DIR / f'{key.lower()}_raw'
            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(extract_dir)
            tif_files = list(extract_dir.rglob('*.tif')) + list(extract_dir.rglob('*.tiff'))
            print(f'  TIF-Dateien: {[f.name for f in tif_files]}')

            if tif_files:
                print('  Clippe auf Donau-Auen BBox...')
                clip_raster_to_bbox(tif_files[0], out_path, BBOX)
                downloaded_files[key] = out_path
            else:
                print('  FEHLER: Kein TIF im ZIP gefunden')

            # Aufraeumen
            zip_path.unlink(missing_ok=True)
            shutil.rmtree(extract_dir, ignore_errors=True)

    except requests.HTTPError as e:
        print(f'CDSE-Auth fehlgeschlagen: {e}')
        print('Pruefen: CDSE_USERNAME und CDSE_PASSWORD in .env')

## 5 — Raster-Metadaten inspizieren mit `rasterio`

Die Funktion `rasterio.open()` liest den GeoTIFF-Header und gibt alle
georeferenzierten Metadaten zurück — **ohne die Rasterdaten in den RAM zu laden**.

Die **Affine Transform** beschreibt mathematisch die Abbildung von Pixel-Koordinaten
auf Weltkoordinaten:

$$\begin{pmatrix} X_{Welt} \\ Y_{Welt} \end{pmatrix} = \begin{pmatrix} X_0 \\ Y_0 \end{pmatrix} + \begin{pmatrix} \Delta x & 0 \\ 0 & \Delta y \end{pmatrix} \begin{pmatrix} Spalte \\ Zeile \end{pmatrix}$$

In [ ]:
def inspect_raster(path: Path, name: str) -> dict:
    """
    Liest Metadaten und Band-1-Daten eines GeoTIFF mit rasterio.

    Gibt ein Dictionary mit CRS, Bounds, Shape, Dtype, NoData und
    deskriptiver Statistik zurueck.
    """
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)  # Band 1 als NumPy-Array
        nd = src.nodata
        if nd is not None:
            data_valid = data[data != nd]
        else:
            data_valid = data.flatten()

        info = {
            'name': name, 'path': path,
            'crs': str(src.crs),
            'bounds': src.bounds,
            'shape': (src.height, src.width),
            'pixel_unit': abs(src.transform.a),
            'dtype': src.dtypes[0],
            'nodata': nd,
            'transform': src.transform,
            'data': data,
            'nodata_val': nd,
            'min': float(data_valid.min()) if len(data_valid) > 0 else None,
            'max': float(data_valid.max()) if len(data_valid) > 0 else None,
            'mean': float(data_valid.mean()) if len(data_valid) > 0 else None,
        }

    print(f'\n{name} ({path.name})')
    print(f'  CRS:     {info["crs"]}')
    print(f'  Shape:   {info["shape"][0]} x {info["shape"][1]} Pixel')
    pu = info['pixel_unit']
    print(f'  Pixel:   {pu:.2f} m/px' if pu < 1 else f'  Pixel:   {pu:.6f} Grad/px (~{pu*111000:.0f}m)')
    print(f'  Bounds:  {info["bounds"]}')
    print(f'  DType:   {info["dtype"]} | NoData: {info["nodata"]}')
    if info['min'] is not None:
        print(f'  Werte:   min={info["min"]:.1f} | mean={info["mean"]:.1f} | max={info["max"]:.1f}')
    t = info['transform']
    print(f'  Affine:  [{t.a:+.4f}, {t.b:+.4f}, {t.c:.2f}]')
    print(f'           [{t.d:+.4f}, {t.e:+.4f}, {t.f:.2f}]')
    return info


# Alle heruntergeladenen Layer inspizieren
raster_infos = {}
for key, path in downloaded_files.items():
    raster_infos[key] = inspect_raster(path, CLMS_LAYERS[key]['description'])

In [ ]:
# Vergleichstabelle
print()
print('ZUSAMMENFASSUNG aller heruntergeladenen CLMS-Layer')
print('-' * 70)
header = f'{"Layer":<5}  {"Shape":>12}  {"Pixel":>10}  {"Min":>5}  {"Max":>5}  {"Mean":>7}'
print(header)
print('-' * 70)
for key, info in raster_infos.items():
    shape_s = f'{info["shape"][0]}x{info["shape"][1]}'
    pu = info['pixel_unit']
    pu_s = f'{pu:.1f}m' if pu < 1 else f'{pu*111000:.0f}m'
    mn = f'{info["min"]:.0f}' if info['min'] is not None else 'N/A'
    mx = f'{info["max"]:.0f}' if info['max'] is not None else 'N/A'
    me = f'{info["mean"]:.1f}' if info['mean'] is not None else 'N/A'
    print(f'{key:<5}  {shape_s:>12}  {pu_s:>10}  {mn:>5}  {mx:>5}  {me:>7}')

## 6 — Visualisierung: Alle Layer nebeneinander

In [ ]:
available = [(key, CLMS_LAYERS[key]) for key in CLMS_LAYERS if key in raster_infos]
n = len(available)

if n == 0:
    print('Keine Layer zum Darstellen.')
else:
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), facecolor='#0d1117')

    # Axes immer als 2D-Liste
    if n == 1:
        axes = [[axes]]
    elif nrows == 1:
        axes = [list(axes)]
    else:
        axes = [list(row) for row in axes]

    fig.suptitle('CLMS-Layer: Nationalpark Donau-Auen (Sprint 2a)',
                 fontsize=15, color='white', fontweight='bold', y=1.01)

    for idx, (key, cfg) in enumerate(available):
        row, col = divmod(idx, ncols)
        ax = axes[row][col]
        ax.set_facecolor('#161b22')

        info = raster_infos[key]
        data = info['data'].copy()
        if info['nodata_val'] is not None:
            data = np.where(data == info['nodata_val'], np.nan, data)

        cmap = plt.get_cmap(cfg['colormap']).copy()
        cmap.set_bad(color='#1c2128')

        b = info['bounds']
        im = ax.imshow(data, cmap=cmap, extent=[b.left, b.right, b.bottom, b.top],
                       origin='upper', interpolation='nearest')

        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(cfg['unit'], color='white', fontsize=8)
        cbar.ax.yaxis.set_tick_params(color='white')
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

        src_label = 'EEA REST' if cfg['source'] == 'eea_rest' else 'CDSE STAC'
        ax.set_title(f'{key} - {cfg["description"].split("(")[0].strip()}',
                     color='white', fontsize=10, fontweight='bold')
        ax.set_xlabel(f'Quelle: {src_label} | {info["shape"][0]}x{info["shape"][1]} px',
                      color='#8b949e', fontsize=8)
        ax.tick_params(colors='#8b949e', labelsize=7)
        for spine in ax.spines.values():
            spine.set_edgecolor('#30363d')

        if info['mean'] is not None:
            stat = f'min={info["min"]:.0f} | mean={info["mean"]:.1f} | max={info["max"]:.0f}'
            ax.text(0.02, 0.02, stat, transform=ax.transAxes, color='white', fontsize=7,
                    bbox=dict(facecolor='#21262d', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.3'))

    for idx in range(n, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row][col].set_visible(False)

    plt.tight_layout()
    out_fig = OUTPUT_DIR / 'clms_sprint2a_overview.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'Abbildung gespeichert: {out_fig}')

## 7 — HTTP Range Requests: COG-Streaming unter der Haube

Jetzt demonstrieren wir den Kern-Mechanismus, der in Sprint 2b von `rioxarray`
automatisch genutzt wird: **HTTP Range Requests gegen einen COG-Endpoint**.

Ein Cloud Optimized GeoTIFF (COG) speichert:
- Daten intern in **Kacheln** (z.B. 512×512 Pixel)
- **Vorschau-Pyramiden** auf niedrigerer Auflösung
- Einen **Kachel-Offset-Index** am Dateianfang

Damit kann ein Client gezielt NUR die Kacheln für eine BBox anfordern.

In [ ]:
# Demo: HTTP Range Request — die Kernidee hinter COG-Streaming
#
# Wir demonstrieren den Range-Request-Mechanismus an einem
# OEFFENTLICHEN EEA-Endpoint (keine Auth noetig).
# prism-dem-open.copernicus.eu ist intern und oft nicht erreichbar.
#
# Ziel: VERSTEHEN was rioxarray.open_rasterio(url) intern macht.

# Wir verwenden den EEA ImageServer als Demo-Endpunkt
# Der 'exportImage'-Endpunkt liefert kein Range-faehiges COG,
# aber das CDSE-Katalog-API liefert uns Links auf echte COGs!
# Wir testen zuerst den EEA-Server auf Range-Support:

EEA_IMG_URL = (
    'https://image.discomap.eea.europa.eu/arcgis/rest/services/'
    'GioLandPublic/HRL_ImperviousnessDensity_2018/ImageServer'
)

print('HTTP-Mechanik Demo')
print('=' * 55)

# HEAD-Request auf EEA ImageServer
r_info = requests.get(EEA_IMG_URL, params={'f': 'json'}, timeout=15)
info = r_info.json()
print(f'EEA ImageServer Info:')
print(f'  ServiceDataType: {info.get("serviceDataType", "n/a")}')
print(f'  CRS (Server): WKID={info.get("spatialReference", {}).get("wkid", "?")} (3857=Web Mercator)')
print(f'  Pixel Size:   {info.get("pixelSizeX", "n/a")} m')
print(f'  noDataValue:  {info.get("noDataValue", "n/a")}')
print()

# HEAD-Request gegen unser eigenes heruntergeladenes Tiff — auf dem Dateisystem
# Zeige wie rasterio intern mit der Datei umgeht
import rasterio
from pathlib import Path

local_tif = OUTPUT_DIR / 'imd_2018_donau_auen.tif'
print(f'Lokale GeoTIFF-Struktur (via rasterio):')
print(f'Datei: {local_tif.name} ({local_tif.stat().st_size/1024:.0f} KB)')
with rasterio.open(local_tif) as src:
    bs = src.block_shapes
    print(f'  Tiled:       {src.profile.get("tiled", False)} (internes Kachel-Raster)')
    print(f'  Block Shape: {bs} px (Kachelgroesse fuer effizientes Lesen)')
    print(f'  Driver:      {src.driver}')
    print(f'  CRS:         {src.crs}  <- EEA liefert EPSG:3857!')
    print(f'  NoData:      {src.nodata}')
    t = src.transform
    print(f'  Pixel-Groesse: {abs(t.a):.2f} m/px (in Web-Mercator-Metern)')

print()
print('WICHTIGE ERKENNTNIS: EEA exportImage CRS-Verhalten')
print('  Der EEA-Server reprojiiziert das Raster automatisch in')
print('  EPSG:3857 (Web Mercator) — passend zur uebergebenen EPSG:4326-BBox.')
print('  In Sprint 2b werden wir alle Layer auf EPSG:3035 bringen (reprojiizieren).')
print()
print('Was rioxarray.open_rasterio(cog_url) intern macht:')
print('  1. HTTP HEAD  -> Dateigroesse und Accept-Ranges pruefen')
print('  2. GET Range(0-16KB) -> COG-Header: Kachel-Offsets und Pyramiden')
print('  3. BBox auf Kacheln mappen -> Welche Kacheln benoetigt?')
print('  4. GET Range(offset_i-offset_j) -> NUR die relevanten Kacheln')
print('  => Ergebnis: <5 MB statt 500 MB fuer die Donau-Auen!')

## 8 — Ökologischer Plausibilitäts-Check

Bevor die Daten in Sprint 3 mit GBIF-Artdaten verschnitten werden,
prüfen wir ob die Werte ökologisch sinnvoll sind.

**Erwartungen Nationalpark Donau-Auen:**
- **IMD (Versiegelung):** Mittelwert < 15% — Schutzgebiet, kaum Bebauung
- **WAW (Feuchtigkeit):** > 5% der Pixel feucht/Wasser — Auencharakter!
- **FTY (Waldtyp):** > 10% Waldbedeckung — Auwald dominant
- **TCD (Kronendichte):** Waldflächen > 20% Überschirmung
- **GRA (Grünland):** > 3% Grünland — Au-Wiesen und Mähwiesen

In [ ]:
print('Oekologischer Plausibilitaets-Check — Nationalpark Donau-Auen')
print('=' * 60)

checks = {
    'IMD': ('Mittelwert < 15% (Schutzgebiet)', lambda d: float(np.nanmean(d)) < 15,
            lambda d: f'Mittelwert = {np.nanmean(d):.1f}%'),
    'WAW': ('> 5% Pixel feucht (Auen!)', lambda d: np.nansum(d > 0) / d.size > 0.05,
            lambda d: f'{np.nansum(d > 0) / d.size * 100:.1f}% feuchte Pixel'),
    'FTY': ('> 10% Wald (Auwald)', lambda d: np.nansum(d > 0) / d.size > 0.10,
            lambda d: f'{np.nansum(d > 0) / d.size * 100:.1f}% Waldpixel'),
    'TCD': ('Waldfleachen Mittelwert > 20%', lambda d: float(np.nanmean(d[d > 0])) > 20 if np.nansum(d>0)>0 else False,
            lambda d: f'{np.nanmean(d[d>0]):.1f}% mittlere Kronendichte' if np.nansum(d>0)>0 else 'keine Waldpixel'),
    'GRA': ('> 3% Gruenland (Au-Wiesen)', lambda d: np.nansum(d == 1) / d.size > 0.03,
            lambda d: f'{np.nansum(d == 1) / d.size * 100:.1f}% Gruenlandpixel'),
}

print()
all_ok = True
for key, (expectation, check_fn, fmt_fn) in checks.items():
    if key not in raster_infos:
        print(f'  [{key}] -- uebersprungen (nicht heruntergeladen)')
        continue
    data = raster_infos[key]['data'].astype(float)
    nd = raster_infos[key]['nodata_val']
    if nd is not None:
        data = np.where(data == nd, np.nan, data)
    try:
        ok = check_fn(data)
        val = fmt_fn(data)
        symbol = 'OK' if ok else 'WARNUNG'
        if not ok:
            all_ok = False
        print(f'  [{key}] {symbol}: {val}')
        print(f'         Erwartung: {expectation}')
    except Exception as e:
        print(f'  [{key}] FEHLER: {e}')

print()
if all_ok:
    print('Alle verfuegbaren Layer plausibel! Bereit fuer Sprint 3 (GBIF-Daten).')
else:
    print('Einige Werte ausserhalb der Erwartung. Pruefen: CRS-Mismatch? NoData?')

## 9 — Zusammenfassung & Ausblick

### Was haben wir in Sprint 2a gelernt?

| Thema | Sprint 2a (manuell) | Sprint 2b (refaktoriert) |
|:---|:---|:---|
| **CLMS-Zugang** | `requests.get(eea_url)` + OData-Download | `rioxarray.open_rasterio(s3_url)` |
| **Datenmenge** | Komplette Kachel ~200-500 MB | Nur Donau-Auen ~5 MB (COG Range) |
| **BBox-Clipping** | `rasterio.read(window=...)` manuell | `ds.rio.clip_box(*bbox)` |
| **CRS EEA** | **EPSG:3857** (Web Mercator) | Alle auf **EPSG:3035** reprojizieren |
| **CRS CDSE** | **EPSG:3035** (LAEA Europe, metrisch) | Konsistentes CRS fuer alle |
| **HTTP-Mechanik** | EEA-Server-Info + Block-Struktur verstanden | GDAL VSICURL macht es automatisch |

### CRS-Hinweis: Was EPSG:3857 bedeutet

Der EEA-Server liefert die Layer in **EPSG:3857 (Web Mercator / Pseudo-Mercator)**:
- Vorteil: Passt direkt zu Web-Kartendiensten (OpenStreetMap, Google Maps)
- Nachteil: **Flaechenverzerrung** bei hoeheren Breiten (Oesterreich: ~10%)
- Fuer **metrische Berechnungen** (Flaechengroesse, Distanzen) immer EPSG:3035!

```python
# Sprint 2b: Reprojektion auf EPSG:3035 fuer alle Layer
import rioxarray as rxr
ds = rxr.open_rasterio('imd_2018_donau_auen.tif')
ds_3035 = ds.rio.reproject('EPSG:3035', resampling='nearest')  # nearest fuer Klassen
ds_3035 = ds.rio.reproject('EPSG:3035', resampling='bilinear') # bilinear fuer % Werte
```

### Erzeugte Dateien

```
data/raw/copernicus/
├── imd_2018_donau_auen.tif     <- Imperviousness (EEA REST, EPSG:3857)
├── waw_2018_donau_auen.tif     <- Water & Wetness (EEA REST, EPSG:3857)
├── fty_2018_donau_auen.tif     <- Forest Type (EEA REST, EPSG:3857)
├── tcd_2024_donau_auen.tif     <- Tree Cover Density (CDSE, EPSG:3035, wenn Auth OK)
├── gra_2024_donau_auen.tif     <- Grassland (CDSE, EPSG:3035, wenn Auth OK)
└── clms_sprint2a_overview.png  <- Visualisierung
```

### Naechster Sprint: 03_gbif_species_data.ipynb

GBIF-Artdaten fuer Feldhase, Fasan, Admiral und Neuntoeter (FFH-Art) —
gefiltert auf Oesterreich, ab 2018, mit GPS-Koordinaten.

In [ ]:
# Abschlussstatus
print('Sprint 2a — Abschlussstatus')
print('=' * 45)
print()

for key, cfg in CLMS_LAYERS.items():
    path = OUTPUT_DIR / cfg['filename']
    src = 'EEA REST' if cfg['source'] == 'eea_rest' else 'CDSE STAC'
    if path.exists():
        kb = path.stat().st_size / 1024
        status = f'OK  ({kb:.0f} KB)'
    else:
        status = 'fehlt (Auth benoetigt)' if cfg['source'] == 'cdse_stac' else 'FEHLT'
    print(f'  {key:<5} [{src}] {status}')

print()
print('Lernziele Sprint 2a:')
print('  OK EEA DiscoMap exportImage-API verstanden & genutzt')
print('  OK CDSE STAC-Katalog abgefragt & Item-Struktur analysiert')
print('  OK OAuth2 ROPC Token-Flow fuer CDSE implementiert')
print('  OK HTTP Range Requests & COG-Mechanismus demonstriert')
print('  OK rasterio-Metadaten & Affine Transform interpretiert')
print('  OK CRS-Unterschiede (EPSG:4326 EEA vs EPSG:3035 CDSE)')
print('  OK Oekologischer Plausibilitaets-Check durchgefuehrt')